# Cell 1 — Install and Imports

In [1]:
!pip -q install pydicom opencv-python-headless albumentations torchvision pyyaml

import os
import cv2
import json
import shutil
import random
import warnings
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import pydicom
from pydicom.pixel_data_handlers.util import apply_voi_lut

warnings.filterwarnings("ignore")
print("Imports done")

Imports done


# Cell 2 - Config, paths, cleanup

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_ROOT = "/kaggle/input/competitions/vinbigdata-chest-xray-abnormalities-detection"
TRAIN_DIR = os.path.join(DATA_ROOT, "train")
TEST_DIR = os.path.join(DATA_ROOT, "test")
TRAIN_CSV = os.path.join(DATA_ROOT, "train.csv")
SAMPLE_SUB = os.path.join(DATA_ROOT, "sample_submission.csv")

print("DATA_ROOT:", DATA_ROOT)

PREP_ROOT = "/kaggle/working/vinbigdata_prep"

# remove old output first so reruns do not keep filling disk
if os.path.exists(PREP_ROOT):
    shutil.rmtree(PREP_ROOT, ignore_errors=True)

CACHE_DIR = os.path.join(PREP_ROOT, "cache_640")
YOLO_ROOT = os.path.join(PREP_ROOT, "yolo640")
YOLO_IMG_TRAIN = os.path.join(YOLO_ROOT, "images", "train")
YOLO_IMG_VAL = os.path.join(YOLO_ROOT, "images", "val")
YOLO_LABEL_TRAIN = os.path.join(YOLO_ROOT, "labels", "train")
YOLO_LABEL_VAL = os.path.join(YOLO_ROOT, "labels", "val")
SPLIT_DIR = os.path.join(PREP_ROOT, "splits")
META_DIR = os.path.join(PREP_ROOT, "meta")

for p in [CACHE_DIR, YOLO_IMG_TRAIN, YOLO_IMG_VAL, YOLO_LABEL_TRAIN, YOLO_LABEL_VAL, SPLIT_DIR, META_DIR]:
    os.makedirs(p, exist_ok=True)

CLASS_NAMES = [
    "Aortic enlargement",
    "Atelectasis",
    "Calcification",
    "Cardiomegaly",
    "Consolidation",
    "ILD",
    "Infiltration",
    "Lung Opacity",
    "Nodule/Mass",
    "Other lesion",
    "Pleural effusion",
    "Pleural thickening",
    "Pneumothorax",
    "Pulmonary fibrosis",
    "No finding",
]
NUM_CLASSES = len(CLASS_NAMES)
NO_FINDING_ID = 14
YOLO_SIZE = 640
CACHE_JPEG_QUALITY = 85  # lower size, still good enough for training

print("Config loaded")

DATA_ROOT: /kaggle/input/competitions/vinbigdata-chest-xray-abnormalities-detection
Config loaded


# Cell 3 — utilities

In [3]:
def ensure_uint8(img):
    img = img.astype(np.float32)
    img = img - img.min()
    mx = img.max()
    if mx > 0:
        img = img / mx
    return (img * 255.0).clip(0, 255).astype(np.uint8)

def read_dicom(path):
    ds = pydicom.dcmread(path)
    img = ds.pixel_array
    try:
        img = apply_voi_lut(img, ds)
    except Exception:
        pass

    img = img.astype(np.float32)

    if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
        img = np.max(img) - img

    if np.ptp(img) > 0:
        img = (img - img.min()) / (img.max() - img.min())
    else:
        img = np.zeros_like(img, dtype=np.float32)

    return (img * 255.0).clip(0, 255).astype(np.uint8)

def read_image_any(path):
    ext = Path(path).suffix.lower()
    if ext in [".dcm", ".dicom"]:
        return read_dicom(path)
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Unable to read image: {path}")
    return img

def letterbox(image, new_shape=640, color=114):
    shape = image.shape[:2]
    if isinstance(new_shape, int):
        new_shape = (new_shape, new_shape)

    ratio = min(new_shape[0] / shape[0], new_shape[1] / shape[1])
    new_unpad = (int(round(shape[1] * ratio)), int(round(shape[0] * ratio)))
    dw = new_shape[1] - new_unpad[0]
    dh = new_shape[0] - new_unpad[1]
    dw /= 2
    dh /= 2

    if shape[::-1] != new_unpad:
        image = cv2.resize(image, new_unpad, interpolation=cv2.INTER_LINEAR)

    top, bottom = int(round(dh - 0.1)), int(round(dh + 0.1))
    left, right = int(round(dw - 0.1)), int(round(dw + 0.1))

    image = cv2.copyMakeBorder(image, top, bottom, left, right, cv2.BORDER_CONSTANT, value=color)
    return image, ratio, (dw, dh)

def iou_xyxy(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter_w = max(0, x2 - x1)
    inter_h = max(0, y2 - y1)
    inter = inter_w * inter_h

    area1 = max(0, box1[2] - box1[0]) * max(0, box1[3] - box1[1])
    area2 = max(0, box2[2] - box2[0]) * max(0, box2[3] - box2[1])
    union = area1 + area2 - inter + 1e-9
    return inter / union

def clip_box(box, w, h):
    x1, y1, x2, y2 = box
    x1 = max(0, min(w - 1, x1))
    y1 = max(0, min(h - 1, y1))
    x2 = max(0, min(w - 1, x2))
    y2 = max(0, min(h - 1, y2))
    if x2 <= x1:
        x2 = min(w - 1, x1 + 1)
    if y2 <= y1:
        y2 = min(h - 1, y1 + 1)
    return [x1, y1, x2, y2]

def xyxy_to_yolo(x1, y1, x2, y2, w, h):
    cx = ((x1 + x2) / 2) / w
    cy = ((y1 + y2) / 2) / h
    bw = (x2 - x1) / w
    bh = (y2 - y1) / h
    return cx, cy, bw, bh

def merge_boxes_same_class(boxes, iou_thr=0.5, min_area=8.0):
    grouped = defaultdict(list)
    for b in boxes:
        grouped[int(b["class_id"])].append(b)

    merged = []
    for cls_id, items in grouped.items():
        items = sorted(items, key=lambda x: x.get("score", 1.0), reverse=True)
        used = [False] * len(items)

        for i in range(len(items)):
            if used[i]:
                continue

            b0 = items[i]["bbox"]
            area0 = max(0, b0[2] - b0[0]) * max(0, b0[3] - b0[1])
            if area0 < min_area:
                used[i] = True
                continue

            cluster = [items[i]]
            used[i] = True

            for j in range(i + 1, len(items)):
                if used[j]:
                    continue
                if iou_xyxy(items[i]["bbox"], items[j]["bbox"]) >= iou_thr:
                    cluster.append(items[j])
                    used[j] = True

            arr = np.array([c["bbox"] for c in cluster], dtype=np.float32)
            merged.append({"class_id": cls_id, "bbox": arr.mean(axis=0).tolist()})

    return merged

def find_src_path(img_id, base_dir):
    candidates = [
        os.path.join(base_dir, f"{img_id}.dicom"),
        os.path.join(base_dir, f"{img_id}.dcm"),
        os.path.join(base_dir, f"{img_id}.png"),
        os.path.join(base_dir, f"{img_id}.jpg"),
        os.path.join(base_dir, f"{img_id}.jpeg"),
    ]
    return next((p for p in candidates if os.path.exists(p)), None)

def hardlink_or_copy(src, dst):
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if os.path.exists(dst):
        os.remove(dst)
    try:
        os.link(src, dst)
    except Exception:
        shutil.copy2(src, dst)

def save_cached_jpg(src_path, out_path):
    img = read_image_any(src_path)
    img = ensure_uint8(img)

    # light contrast cleanup
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img = clahe.apply(img)

    # one standard 640 cache used everywhere
    img = cv2.resize(img, (YOLO_SIZE, YOLO_SIZE), interpolation=cv2.INTER_AREA)
    img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    cv2.imwrite(out_path, img, [int(cv2.IMWRITE_JPEG_QUALITY), CACHE_JPEG_QUALITY])
    return out_path

print("Utilities ready")

Utilities ready


# Cell 4 — Load annotations and build label maps

In [4]:
df = pd.read_csv(TRAIN_CSV)
print(df.head())
print("Rows:", len(df))

image_groups = df.groupby("image_id")
image_label_map = {}

for img_id, g in image_groups:
    cls_ids = set(g["class_id"].astype(int).tolist())
    image_label_map[img_id] = {
        "has_abnormal": any(c != NO_FINDING_ID for c in cls_ids),
        "has_no_finding": (NO_FINDING_ID in cls_ids) and len(cls_ids) == 1,
        "classes": sorted(cls_ids),
    }

ann_per_image = defaultdict(list)
for _, row in df.iterrows():
    img_id = row["image_id"]
    cls_id = int(row["class_id"])
    if cls_id == NO_FINDING_ID:
        continue
    if pd.isna(row["x_min"]) or pd.isna(row["y_min"]) or pd.isna(row["x_max"]) or pd.isna(row["y_max"]):
        continue
    ann_per_image[img_id].append({
        "class_id": cls_id,
        "class_name": row["class_name"],
        "bbox": [float(row["x_min"]), float(row["y_min"]), float(row["x_max"]), float(row["y_max"])],
        "rad_id": row["rad_id"],
    })

fused_ann_per_image = {}
for img_id, boxes in ann_per_image.items():
    fused_ann_per_image[img_id] = merge_boxes_same_class(boxes, iou_thr=0.5)

print("Images with annotations:", len(fused_ann_per_image))
print("Example:", next(iter(fused_ann_per_image.items())) if fused_ann_per_image else "None")

                           image_id          class_name  class_id rad_id  \
0  50a418190bc3fb1ef1633bf9678929b3          No finding        14    R11   
1  21a10246a5ec7af151081d0cd6d65dc9          No finding        14     R7   
2  9a5094b2563a1ef3ff50dc5c7ff71345        Cardiomegaly         3    R10   
3  051132a778e61a86eb147c7c6f564dfe  Aortic enlargement         0    R10   
4  063319de25ce7edb9b1c6b8881290140          No finding        14    R10   

    x_min   y_min   x_max   y_max  
0     NaN     NaN     NaN     NaN  
1     NaN     NaN     NaN     NaN  
2   691.0  1375.0  1653.0  1831.0  
3  1264.0   743.0  1611.0  1019.0  
4     NaN     NaN     NaN     NaN  
Rows: 67914
Images with annotations: 4394
Example: ('9a5094b2563a1ef3ff50dc5c7ff71345', [{'class_id': 3, 'bbox': [690.6666870117188, 1354.3333740234375, 1658.6666259765625, 1797.6666259765625]}, {'class_id': 10, 'bbox': [1789.0, 1729.0, 1875.0, 1992.0]}, {'class_id': 11, 'bbox': [1789.0, 1729.0, 1875.0, 1992.0]}, {'class_id':

# Cell 5 — build the single image cache only once

In [5]:
def build_cache_640(force_rebuild=False):
    records = []
    img_ids = sorted(df["image_id"].unique().tolist())

    for img_id in tqdm(img_ids, desc="Building 640 cache"):
        src_path = find_src_path(img_id, TRAIN_DIR)
        if src_path is None:
            continue

        cache_path = os.path.join(CACHE_DIR, f"{img_id}.jpg")

        try:
            if force_rebuild or (not os.path.exists(cache_path)):
                save_cached_jpg(src_path, cache_path)

            records.append({
                "image_id": img_id,
                "src_path_rel": os.path.relpath(src_path, PREP_ROOT) if src_path.startswith(PREP_ROOT) else src_path,
                "cache_path_rel": os.path.relpath(cache_path, PREP_ROOT),
                "has_abnormal": int(image_label_map.get(img_id, {}).get("has_abnormal", False)),
                "has_no_finding": int(image_label_map.get(img_id, {}).get("has_no_finding", False)),
                "classes": " ".join(map(str, image_label_map.get(img_id, {}).get("classes", []))),
            })
        except Exception as e:
            print(f"Failed on {img_id}: {e}")

    return pd.DataFrame(records)

cache_df = build_cache_640(force_rebuild=False)
print(cache_df.head())
print("Cached images:", len(cache_df))

cache_df.to_csv(os.path.join(PREP_ROOT, "cache_df.csv"), index=False)
cache_df.to_csv(os.path.join(META_DIR, "cache_df.csv"), index=False)

Building 640 cache:   0%|          | 0/15000 [00:00<?, ?it/s]

                           image_id  \
0  000434271f63a053c4128a0ba6352c7f   
1  00053190460d56c53cc3e57321387478   
2  0005e8e3701dfb1dd93d53e2ff537b6e   
3  0006e0a85696f6bb578e84fafa9a5607   
4  0007d316f756b3fa0baea2ff514ce945   

                                        src_path_rel  \
0  /kaggle/input/competitions/vinbigdata-chest-xr...   
1  /kaggle/input/competitions/vinbigdata-chest-xr...   
2  /kaggle/input/competitions/vinbigdata-chest-xr...   
3  /kaggle/input/competitions/vinbigdata-chest-xr...   
4  /kaggle/input/competitions/vinbigdata-chest-xr...   

                                   cache_path_rel  has_abnormal  \
0  cache_640/000434271f63a053c4128a0ba6352c7f.jpg             0   
1  cache_640/00053190460d56c53cc3e57321387478.jpg             0   
2  cache_640/0005e8e3701dfb1dd93d53e2ff537b6e.jpg             1   
3  cache_640/0006e0a85696f6bb578e84fafa9a5607.jpg             0   
4  cache_640/0007d316f756b3fa0baea2ff514ce945.jpg             1   

   has_no_finding      cl

# Cell 6 — Train/Val split

In [6]:
from sklearn.model_selection import train_test_split

train_ids, val_ids = train_test_split(
    cache_df["image_id"].tolist(),
    test_size=0.2,
    random_state=SEED,
    stratify=cache_df["has_abnormal"].tolist() if len(cache_df["has_abnormal"].unique()) > 1 else None,
)

train_df_split = cache_df[cache_df["image_id"].isin(train_ids)].reset_index(drop=True)
val_df_split = cache_df[cache_df["image_id"].isin(val_ids)].reset_index(drop=True)

train_df_split.to_csv(os.path.join(SPLIT_DIR, "train_split.csv"), index=False)
val_df_split.to_csv(os.path.join(SPLIT_DIR, "val_split.csv"), index=False)

print("Train:", len(train_df_split), "Val:", len(val_df_split))

Train: 12000 Val: 3000


# Cell 7 — Prepare YOLO folders with hardlinks

In [7]:
def prepare_yolo_split(records_df, split_name):
    if split_name not in ["train", "val"]:
        raise ValueError("split_name must be train or val")

    img_out_dir = YOLO_IMG_TRAIN if split_name == "train" else YOLO_IMG_VAL
    lbl_out_dir = YOLO_LABEL_TRAIN if split_name == "train" else YOLO_LABEL_VAL

    img_list = []

    for _, row in tqdm(records_df.iterrows(), total=len(records_df), desc=f"Preparing {split_name}"):
        img_id = row["image_id"]
        src_path = os.path.join(PREP_ROOT, row["src_path_rel"]) if not str(row["src_path_rel"]).startswith("/kaggle/input") else row["src_path_rel"]
        cache_path = os.path.join(PREP_ROOT, row["cache_path_rel"])

        yolo_img_path = os.path.join(img_out_dir, f"{img_id}.jpg")
        lbl_path = os.path.join(lbl_out_dir, f"{img_id}.txt")

        try:
            # hardlink the single cached JPG into YOLO folder
            hardlink_or_copy(cache_path, yolo_img_path)
            img_list.append(os.path.abspath(yolo_img_path))

            # original size for label conversion
            orig_img = read_image_any(src_path)
            orig_img = ensure_uint8(orig_img)
            orig_h, orig_w = orig_img.shape[:2]

            ratio = min(YOLO_SIZE / orig_h, YOLO_SIZE / orig_w)
            new_w = int(round(orig_w * ratio))
            new_h = int(round(orig_h * ratio))
            dw = (YOLO_SIZE - new_w) / 2
            dh = (YOLO_SIZE - new_h) / 2

            boxes = fused_ann_per_image.get(img_id, [])
            lines = []

            for b in boxes:
                cls_id = int(b["class_id"])
                if cls_id == NO_FINDING_ID:
                    continue

                x1, y1, x2, y2 = b["bbox"]
                x1 = x1 * ratio + dw
                x2 = x2 * ratio + dw
                y1 = y1 * ratio + dh
                y2 = y2 * ratio + dh

                x1 = max(0, min(YOLO_SIZE - 1, x1))
                y1 = max(0, min(YOLO_SIZE - 1, y1))
                x2 = max(0, min(YOLO_SIZE - 1, x2))
                y2 = max(0, min(YOLO_SIZE - 1, y2))

                if x2 <= x1 or y2 <= y1:
                    continue

                cx, cy, bw, bh = xyxy_to_yolo(x1, y1, x2, y2, YOLO_SIZE, YOLO_SIZE)
                if bw <= 0 or bh <= 0:
                    continue

                lines.append(f"{cls_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")

            with open(lbl_path, "w") as f:
                f.write("\n".join(lines))

        except Exception as e:
            print(f"Failed on {img_id}: {e}")

    return img_list

train_img_list = prepare_yolo_split(train_df_split, "train")
val_img_list = prepare_yolo_split(val_df_split, "val")

with open(os.path.join(YOLO_ROOT, "train.txt"), "w") as f:
    f.write("\n".join(train_img_list))

with open(os.path.join(YOLO_ROOT, "val.txt"), "w") as f:
    f.write("\n".join(val_img_list))

print("YOLO train/val lists written")

Preparing train:   0%|          | 0/12000 [00:00<?, ?it/s]

Preparing val:   0%|          | 0/3000 [00:00<?, ?it/s]

YOLO train/val lists written


# Cell 8 — Create YOLO yaml

In [8]:
def create_yolo_yaml(yaml_path):
    data = {
        "path": PREP_ROOT,
        "train": "yolo640/train.txt",
        "val": "yolo640/val.txt",
        "names": {i: n for i, n in enumerate(CLASS_NAMES[:-1])},
    }
    with open(yaml_path, "w") as f:
        import yaml
        yaml.dump(data, f, sort_keys=False)

YOLO_YAML = os.path.join(YOLO_ROOT, "vinbigdata_640.yaml")
create_yolo_yaml(YOLO_YAML)

print("YOLO YAML saved to:", YOLO_YAML)
print(open(YOLO_YAML).read())

YOLO YAML saved to: /kaggle/working/vinbigdata_prep/yolo640/vinbigdata_640.yaml
path: /kaggle/working/vinbigdata_prep
train: yolo640/train.txt
val: yolo640/val.txt
names:
  0: Aortic enlargement
  1: Atelectasis
  2: Calcification
  3: Cardiomegaly
  4: Consolidation
  5: ILD
  6: Infiltration
  7: Lung Opacity
  8: Nodule/Mass
  9: Other lesion
  10: Pleural effusion
  11: Pleural thickening
  12: Pneumothorax
  13: Pulmonary fibrosis



# Cell 9 — Save metadata

In [9]:
meta = {
    "prep_root": PREP_ROOT,
    "data_root": DATA_ROOT,
    "cache_dir": CACHE_DIR,
    "yolo_root": YOLO_ROOT,
    "yolo_yaml": YOLO_YAML,
    "num_images": len(cache_df),
    "num_train": len(train_df_split),
    "num_val": len(val_df_split),
    "class_names": CLASS_NAMES,
    "no_finding_id": NO_FINDING_ID,
    "jpeg_quality": CACHE_JPEG_QUALITY,
}

with open(os.path.join(PREP_ROOT, "prep_meta.json"), "w") as f:
    json.dump(meta, f, indent=2)

print(json.dumps(meta, indent=2))

{
  "prep_root": "/kaggle/working/vinbigdata_prep",
  "data_root": "/kaggle/input/competitions/vinbigdata-chest-xray-abnormalities-detection",
  "cache_dir": "/kaggle/working/vinbigdata_prep/cache_640",
  "yolo_root": "/kaggle/working/vinbigdata_prep/yolo640",
  "yolo_yaml": "/kaggle/working/vinbigdata_prep/yolo640/vinbigdata_640.yaml",
  "num_images": 15000,
  "num_train": 12000,
  "num_val": 3000,
  "class_names": [
    "Aortic enlargement",
    "Atelectasis",
    "Calcification",
    "Cardiomegaly",
    "Consolidation",
    "ILD",
    "Infiltration",
    "Lung Opacity",
    "Nodule/Mass",
    "Other lesion",
    "Pleural effusion",
    "Pleural thickening",
    "Pneumothorax",
    "Pulmonary fibrosis",
    "No finding"
  ],
  "no_finding_id": 14,
  "jpeg_quality": 85
}


# Cell 10 — Final size check

In [10]:
total_bytes = 0
for root, _, files in os.walk(PREP_ROOT):
    for f in files:
        total_bytes += os.path.getsize(os.path.join(root, f))

print(f"Approx output size: {total_bytes / (1024**3):.2f} GB")
print("If this is around 1–3 GB, you are safely below your 18 GB target.")

Approx output size: 1.92 GB
If this is around 1–3 GB, you are safely below your 18 GB target.


In [11]:
import os
from IPython.display import FileLink

# 1. Zip your folder (the fastest compression method)
# Replace 'your_folder_name' with the actual folder you want to download
!zip -r -q output_data.zip /kaggle/working/vinbigdata_prep

# 2. Generate and display the fast download link
FileLink(r'output_data.zip')

/kaggle/working/output_data.zip